# Chapter 03: Data Preprocessing

## Engineering Question
> Why must data preprocessing transformations be fitted strictly on the training set and serialized for inference, and what are the visual and statistical effects of standardization on network features?

---

### Objective
The objective of this notebook is to execute our standardized preprocessing pipeline on the raw NSL-KDD dataset using our modular backend (`src.data.preprocessing`). We will split features and target columns, encode categorical values, binarize target labels, apply standard scaling, and serialize the fitted transform objects. We will also visualize feature distribution shifts before and after scaling to confirm standardization correctness.

## Background / Theory

### Why Preprocessing Matters
Raw network traffic logs contain a mix of string states (e.g. `tcp`, `http`, `SF`) and numerical attributes spanning vast scales (e.g. `src_bytes` from $0$ to billions). Standard machine learning algorithms expect a clean, normalized matrix of float values. Preprocessing transforms raw logs into model-ready structures while ensuring consistency.

### Label Encoding vs. One-Hot Encoding
In this project, we utilize `LabelEncoder` rather than `OneHotEncoder` for categorical columns:
- **High Cardinality**: The `service` feature contains over 70 unique values (e.g. `private`, `http`, `ftp_data`). One-hot encoding this would append 70+ sparse columns to the feature matrix.
- **Curse of Dimensionality**: Adding 70 sparse features exacerbates dimensionality issues, degrading the performance of distance-based estimators (DBSCAN and Autoencoders).
- **Tree splits**: Isolation Forest works by making random axis-aligned splits. Splitting on sparse, binary one-hot features creates deep, unbalanced trees. Label encoding represents categories densely, allowing more efficient splits.

### StandardScaler vs. MinMaxScaler
- **MinMaxScaler**: Maps data to the range `[0, 1]` based on maximum and minimum values. If a feature contains extreme values (e.g. packet counts with massive outliers), normalization squashes the normal variance into a narrow band, destroying structural details.
- **StandardScaler**: Standardizes features by removing the mean and scaling to unit variance ($z$-score). This preserves the underlying variance scale and handles outliers robustly.

### Preventing Data Leakage
A critical rule in machine learning engineering is: **never fit scalers or encoders on the entire dataset (or test set)**. Encoders and scalers must be fit strictly on the training set. Fitting on the test set leaks information about the global mean, standard deviation, and category vocabulary. This results in overly optimistic test scores and model failure during real-world inference. After fitting, transform parameters must be serialized (saved as `.joblib` files) so the same offsets can be applied to unseen validation and inference records.

## Preprocessing Pipeline

```text
  [Raw Training DataFrame] (125973 rows)
              │
              ▼
  [encode_categorical_features] ──► Integer Map (Fitted on Train)
              │
              ▼
  [split_features_and_target] ────► X (Features), y (Label Series)
              │
       ┌──────┴──────┐
       ▼             ▼
  [fit_scaler]  [get_binary_labels]
       │             │
       ▼             ▼
  X_scaled      y_binary (normal -> 0, attacks -> 1)
       │             │
       ▼             ▼
 [Save Scaler]  [Clean Arrays ready for Training]
```

## Imports

All imports originate from standard libraries, Plotly, or our modularized project backend (`src` / `configs`).

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

# Ensure project root is in path for imports
sys.path.append(os.path.abspath(".." if ".." in sys.path else ".."))

from configs import config
from src.data.dataset import load_train_data
from src.data.preprocessing import (
    encode_categorical_features,
    split_features_and_target,
    fit_scaler,
    get_binary_labels,
    prepare_training_data
)

# Set Plotly default template
pio.templates.default = config.PLOT_TEMPLATE

## Preprocessing Pipeline Execution

We load the raw training dataset and run the transformation steps sequentially. This ensures we fit and save the encoders and scalers.

In [2]:
# Step 1: Load Data
raw_df = load_train_data()
print(f"1. Dataset loaded. Shape: {raw_df.shape}")

# Step 2: Encode Categories
encoded_df, encoders = encode_categorical_features(raw_df)
print("2. Categorical features encoded successfully.")

# Step 3: Split Features/Target
x, y = split_features_and_target(encoded_df)
print(f"3. Features and targets split. Features shape: {x.shape}")

# Step 4: Fit and Save Scaler
scaler, x_scaled = fit_scaler(x)
print(f"4. Scaler fitted and serialized successfully. Scaled features shape: {x_scaled.shape}")

# Step 5: Convert Target to Binary
y_binary = get_binary_labels(y)
print(f"5. Targets converted to binary labels. Value counts:\n{y_binary.value_counts()}")

1. Dataset loaded. Shape: (125973, 42)


2. Categorical features encoded successfully.
3. Features and targets split. Features shape: (125973, 41)


4. Scaler fitted and serialized successfully. Scaled features shape: (125973, 41)
5. Targets converted to binary labels. Value counts:
label
0    67343
1    58630
Name: count, dtype: int64


## Visualizing Feature Distributions Before and After Scaling

Let's select a continuous numerical feature (e.g. `count` - the number of connections to the same host in the past two seconds) and visualize its distribution before and after applying standard scaling. This illustrates the visual and statistical effects of standardization.

In [3]:
# Distribution before scaling
fig_before = px.histogram(
    x.head(2000),
    x='count',
    title='Count Feature Distribution BEFORE Scaling (Subset)',
    labels={'x': 'Raw Count Value'},
    nbins=50
)
fig_before.update_layout(width=650, height=400)
fig_before.show()

Let's look at the distribution of the same feature after standardization.

In [4]:
# Distribution after scaling
fig_after = px.histogram(
    x_scaled.head(2000),
    x='count',
    title='Count Feature Distribution AFTER Standardization (Subset)',
    labels={'x': 'Standardized z-score'},
    nbins=50
)
fig_after.update_layout(width=650, height=400)
fig_after.show()

Verify that our centralized orchestrator wrapper `prepare_training_data` reproduces the exact same output shape and properties.

In [5]:
x_orch, y_orch = prepare_training_data(raw_df)

shapes_match = (x_orch.shape == x_scaled.shape) and (y_orch.shape == y.shape)
print(f"Orchestrator output shapes match components: {shapes_match}")
print(f"Feature matrices mean: count={x_orch['count'].mean():.3f}, srv_count={x_orch['srv_count'].mean():.3f} (scaled to 0 mean)")

Orchestrator output shapes match components: True
Feature matrices mean: count=-0.000, srv_count=0.000 (scaled to 0 mean)


## Engineering Notes

### Reproducibility and Serialization
By saving encoders and scalers in `outputs/encoders/` and `outputs/scalers/`, we guarantee that validation or production scripts load the identical transformations. Real-world packet ingestion must reuse the same scale parameters, otherwise z-scores will be calculated incorrectly, breaking prediction thresholds.

### Data Leakage in Practice
If we scale our data before splitting it or fit a scaler on test records, the test mean leaks into the training phase. This leads to artificial over-fitting, where the model behaves well in development but crashes during production drift.

## Interview Questions

1. **What is data leakage in machine learning, and how does standard scaling the entire dataset before splitting introduce it?**
   * *Guideline*: Define data leakage as exposing test set parameters to the training step. Scaling the combined dataset computes global means and variances, leaking test distribution profiles, causing overly optimistic validation metrics.

2. **Why is OneHotEncoding unsuitable for high-cardinality features like network service, and how does LabelEncoding help?**
   * *Guideline*: One-hot encoding 70+ unique services generates 70+ sparse dimensions. This increases distance-metric complexity (Curse of Dimensionality) and makes decision trees very sparse. LabelEncoding maps them to a single dense column.

3. **Why should MinMaxScaler be avoided for features with long-tailed distributions or extreme outliers?**
   * *Guideline*: MinMaxScaler maps values strictly to `[0, 1]` using min/max bounds. Outliers widen these bounds, forcing normal values to cluster in a tiny range (e.g. `[0, 0.01]`), destroying feature variance. StandardScaler scales relative to standard deviation, preserving variance.

4. **How do tree models (like Isolation Forest) handle unscaled data compared to neural networks (like Autoencoders)?**
   * *Guideline*: Decision trees make splits along single axes (axis-aligned splits) and are monotonic, making them scale-invariant. Neural networks compute gradients based on weight multiplications across all features, requiring standardized ranges to prevent gradient explosion or vanishing.

5. **During online inference, a service is encountered that was not present in the training vocabulary. How should your preprocessing pipeline handle this?**
   * *Guideline*: The encoder should catch the unseen value, log a warning, and map it to a default token or placeholder (e.g. `'other'`). Re-fitting the encoder on inference data would alter category-to-integer mappings, corrupting model inputs.

6. **Why is it important to save encoders and scalers using serialization libraries (like joblib or pickle) rather than retraining them?**
   * *Guideline*: Serializing preserves the exact offsets (means, variances, vocabulary maps). Retraining on inference data shifts these parameters, changing inputs and degrading predictions.

7. **How does standard scaling impact the reconstruction loss calculation in an Autoencoder model?**
   * *Guideline*: Autoencoders use reconstruction loss (like Mean Squared Error). Without scaling, features with large scales (e.g. packet bytes) dominate the loss calculation, forcing the network to optimize solely for those columns while ignoring packet rates.

8. **Under what conditions would OneHotEncoding be preferred over LabelEncoding?**
   * *Guideline*: One-hot encoding is preferred for low-cardinality features ($<5$ unique categories) where there is no ordinal relationship, preventing distance-based models from assuming category integers represent ordering.

## Key Takeaways
- Fitted transforms must be strictly saved to prevent leakage.
- Categorical values must be mapped to dense integers to prevent sparse dimensional growth.
- Standardization transforms continuous inputs to zero mean and unit variance.

## Future Improvements
- **Robust Scaler Options**: Introduce option to use a `RobustScaler` (which uses IQR instead of standard deviation) in `configs/config.py` to handle extreme network spikes without skewing mean scores.

## Conclusion

We have preprocessed the training dataset, standardized feature ranges, binarized targets, and saved our scaler and encoders. The dataset is now ready for model training.

## Next Notebook

Proceed to the next chapter: [Isolation Forest Model Training](file:///c:/Projects/Network%20anomoly%20detection/notebooks/04_isolation_forest.ipynb)